# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guided walkthrough for loading and exploring the FAIR² dataset using the `mlcroissant` library, following the Croissant metadata schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name if hasattr(metadata, 'name') else metadata.get('name')}")
print(f"Description: {metadata.description if hasattr(metadata, 'description') else metadata.get('description')}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s (identifiers).

*Each entity is referenced by their `@id` attribute as required by the Croissant schema.*

In [ ]:
# Inspect available record sets in the dataset

record_sets = dataset.metadata.record_sets
print(f"Total record sets found: {len(record_sets)}\n")

for rs in record_sets:
    print(f"Record Set Name: {rs.name}, @id: {rs.id if hasattr(rs, 'id') else getattr(rs, '@id', None)}")
    print(f"  Description: {rs.description if hasattr(rs, 'description') else getattr(rs, 'description', None)}")
    if hasattr(rs, 'fields'):
        print("  Fields and their @id's:")
        for field in rs.fields:
            print(f"    - {field.name} (@id: {field.id if hasattr(field, 'id') else getattr(field, '@id', None)})")
    print()

## 3. Data Extraction
Load data from specific record sets into pandas DataFrames using their Croissant `@id` fields as references.

*For the FAIR² dataset, you can use the printed record set and field IDs from above. Adjust the IDs below to match those printed above if needed.*

In [ ]:
# Collect all record set @ids
record_set_ids = [rs.id if hasattr(rs,'id') else getattr(rs,'@id', None) for rs in dataset.metadata.record_sets]

# For demonstration, we'll use the first record set for extraction (adjust the index if needed)
main_record_set_id = record_set_ids[0]
print(f"Selected record set @id: {main_record_set_id}\n")
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for {record_set_id} ...")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) > 0:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"  Loaded {len(df)} rows and {len(df.columns)} columns.")
        else:
            print("  No records found.")
    except Exception as ex:
        print(f"  Error loading records: {ex}")
print("\nColumns for the main record set:")
print(dataframes[main_record_set_id].columns.tolist())

# Display head for the first record set
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's process and analyze the main record set DataFrame. We will select a numeric field (e.g., protein abundance or MW), filter records, normalize the chosen numeric field, and optionally group by a categorical identifier (such as protein description or accession).


In [ ]:
df = dataframes[main_record_set_id]

# List all numeric-like columns for demonstration:
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
print(f"Numeric columns in selected record set: {numeric_cols}")

# For this dataset, let's assume 'MW' (Molecular Weight) or a protein abundance column exists.
# You may need to adjust the field name by checking the columns above.
candidate_fields = [col for col in numeric_cols if 'weight' in col.lower() or 'abundance' in col.lower() or 'mw' == col.lower()]
if candidate_fields:
    numeric_field = candidate_fields[0]
else:
    # fallback to the first numeric field
    numeric_field = numeric_cols[0] if numeric_cols else None

if numeric_field:
    print(f"\nSelected numeric field for analysis: {numeric_field}\n")
    # Apply a filter (e.g., keep proteins with MW > 10000)
    threshold = df[numeric_field].quantile(0.7) if df[numeric_field].dtype.kind in 'fi' else 10
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}: {len(filtered_df)} rows\n")
    
    # Normalize the numeric field (Z-score)
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Top 5 rows with normalized {numeric_field}:\n")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a descriptive field
    group_field_candidates = [col for col in df.columns if any(k in col.lower() for k in ['protein', 'description', 'accession'])]
    if group_field_candidates:
        group_field = group_field_candidates[0]
        print(f"Grouping by: {group_field}\n")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame('mean_value')
        print(grouped_df.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize field distributions and relationships using matplotlib and seaborn. Adjust field names as needed.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field], kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot by group if group_field is defined
    if 'group_field' in locals():
        plt.figure(figsize=(12, 4))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=90)
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated a reproducible and modular workflow for loading, exploring, and processing the FAIR² mass spectrometry dataset using `mlcroissant` and the Croissant schema.

* The dataset allows programmatic referencing of entities by their Croissant `@id` fields for robust and transparent processing.
* We examined available record sets, loaded data, filtered and normalized numerical values, and visualized key distributions.

You can now extend this analysis with domain-specific modeling, deeper aggregations, or other analytical tasks!